[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/14-planning.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 14. Planning

A multi-step customer question gets an explicit plan before any tool runs.

`draft_plan` is one Chat Completions call with an empty tool list. It returns a short numbered list. `run_agent` prints that plan, attaches it to the system message, and only then starts perceive, reason, act, observe. The plan cannot add a tool. Charge, cancel, and email are not steps it is allowed to invent.

The next sections define the two functions. `plan_steps` keeps the numbered lines. `draft_plan` writes the plan and registers the hook the loop calls.

The customer is Maya. She wants a jar she can eat, and she wants to know if it ships to Ohio. Preference, the catalog, and shipping are separate steps. The plan names the order. The loop still executes only registered tools.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Numbered steps

`plan_steps`, defined in the next cell, reads a finished plan and returns only the numbered lines.

The model is asked for 3 to 6 steps and then the line `PLAN_READY`. This function keeps the steps, in order, and leaves the rest out. The run below prints that list and checks that it has at least two lines.

1. **Split the text into lines.** `None` and an empty string produce no lines, so the result is `[]`.
2. **Strip each line.** Skip a line that is empty after stripping.
3. **Keep a line whose first character is a digit and whose number ends with `.` or `)`.** `1. get_preference allergy` is a step. `2) query_catalog chili` is a step. `10. get_store_fact shipping` is a step, because the second character is still a digit and the third is `.` or `)`.
4. **Drop every other line.** `PLAN_READY` does not start with a digit. A sentence about Ohio does not either. A bare `1` has no `.` or `)` after the number, so it is not a step.
5. **Return those lines unchanged.** The function does not read the tool name. It only decides which lines are steps.

In [ ]:
def plan_steps(text):
    """Return the numbered lines of a plan. Empty when the text is not a list."""
    steps = []
    for line in (text or "").splitlines():
        stripped = line.strip()
        if not stripped or not stripped[0].isdigit():
            continue
        if len(stripped) > 1 and stripped[1] in ".)":
            steps.append(stripped)
        elif len(stripped) > 2 and stripped[1].isdigit() and stripped[2] in ".)":
            steps.append(stripped)
    return steps

## One call, with no tools

`draft_plan`, defined in the next cell, writes the plan. It is one Chat Completions call. The tool list on that call is empty, so the model can name a tool and the call itself cannot run one.

`PLAN_BRIEF` is in the same cell, with a comment, because only this function reads it. The last line of the cell registers the function: `set_hook("draft_plan", draft_plan)`. `attach_plan` in `tutorial.common.loop` looks up that name. Until this cell has run, the loop drafts nothing. This function does not import the loop, and the loop does not import this notebook.

1. **Discard the system argument.** The loop passes the seller's system text. This call writes its own brief, so the argument is unused. `del system` marks that.
2. **Read the tool names registered right now.** They come from `TOOLS`. The plan is allowed to name only those.
3. **Build the brief.** `PLAN_BRIEF` asks for 3 to 6 numbered steps. Each step names one registered tool and the argument it would pass. `get_store_fact` takes a topic. `query_catalog` takes a product name or sku. The reply must not call a tool. It must not name `charge_card`, `cancel_order`, or `send_customer_email`. The plan cannot grant a tool or change a price. The last line is `PLAN_READY`. The function then appends the registered names, or `(none)` when the registry is empty, and tells the model to name only those.
4. **Call `chat`.** The messages are the brief as `system` and the customer's words as `user`. `tools=[]` is the empty list, so this turn cannot act.
5. **Keep the text.** Blank content becomes `(empty plan)`. The function prints `PLAN:` and the text before it returns.
6. **Return the plan and the usage.** The dict has `text`, `prompt_tokens`, `completion_tokens`, and `latency_ms`. `attach_plan` writes `text` onto the system message before any tool runs. When the cost hook is on, it also records that usage as step `plan`. The registry is unchanged: the plan cannot add a tool.
7. **Register the hook.** `set_hook` stores the function under `"draft_plan"`. `run_agent` calls `attach_plan` once, at the start of a fresh thread. Tutorial 15 defines this same function in its own notebook, so that lesson can run alone, and calls `attach_plan` once, on the first real customer line. Later lines in that visit leave the plan where it is.

In [ ]:
from tutorial.common.hooks import set_hook
from tutorial.common.tools import TOOLS
from tutorial.runtime import chat

# Only draft_plan reads this brief.
PLAN_BRIEF = (
    "You write a short plan for the Harbor Jar seller. "
    "The person asking is a customer on the store website. "
    "Reply with 3 to 6 numbered steps. "
    "Each step names one registered tool and the argument you would pass. "
    "get_store_fact takes a topic, never a filename. "
    "query_catalog takes a product name or sku, never SQL. "
    "Do not call any tool in this reply. "
    "Do not name charge_card, cancel_order, or send_customer_email. "
    "The plan cannot grant tools or change prices. "
    "End with the line PLAN_READY."
)


def draft_plan(user_text, system):
    """One Chat Completions call with no tools. Returns the plan and its usage."""
    del system
    names = [tool["function"]["name"] for tool in TOOLS]
    allowed = ", ".join(names) if names else "(none)"
    brief = PLAN_BRIEF + " Registered tools right now: " + allowed + ". Name only those."
    turned = chat(
        [
            {"role": "system", "content": brief},
            {"role": "user", "content": user_text},
        ],
        tools=[],
    )
    text = (turned["content"] or "").strip() or "(empty plan)"
    print("PLAN:")
    print(text)
    print()
    return {
        "text": text,
        "prompt_tokens": turned["usage"]["prompt_tokens"],
        "completion_tokens": turned["usage"]["completion_tokens"],
        "latency_ms": turned["latency_ms"],
    }


# attach_plan in loop.py calls this name. Registering here turns the plan on.
set_hook("draft_plan", draft_plan)

## Plan, then act

`plan_steps` and `draft_plan` are defined above, and `draft_plan` is registered. This cell imports the earlier tools so the plan has real names to use. Maya asks for a jar she can eat and whether it ships to Ohio. `run_agent` drafts the plan, attaches it to the system message, and only then starts the loop.

In [ ]:
import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.memory
import tutorial.common.skills

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_db()
question = "I'm Maya. Please recommend a jar I can eat, and tell me whether you can ship it to Ohio."
result = run_agent(question, system_text(), max_steps=8)
print("STEPS:")
for step in plan_steps(result["plan"]):
    print(step)
print("ANSWER:", result["text"])
steps = plan_steps(result["plan"])
log = "\n".join(result["tool_log"])
check(len(steps) >= 2, "the plan is an ordered list written before the tools")
check("charge_card" not in result["plan"], "the plan does not add a card charge")
check(bool(result["tool_log"]), "the loop ran tools after the plan")
check("get_preference" in log or "preference" in result["plan"].lower(), "Maya's preference is part of the work")
check("query_catalog" in log or "get_store_fact" in log, "store data was read while following the plan")